# Guarded Personalization — contamination-aware continual anomaly detection
**One-page brief → runnable code.** Implements the pipeline from `Guarded_Personalization_one_page.docx`:
personal TimesFM+LoRA detector → LLM with three roles (context reader, safety blocker, PBRS prior)
→ RL agent (`ALERT` / `ADAPT` / `WAIT` / `ASK`) → independent conformal shield → gated model update.

**Dataset choice.** The best runnable option today is the *controlled simulator*: real healthy
traces (Stanford smartwatch zip when `STANFORD_ZIP` points at it, synthetic fallback otherwise)
+ injected, labelled illness episodes and benign shifts. Nothing illness-labelled is in hand yet
(Evidation: Synapse request pending; CovIdentify: PhysioNet credentialing in progress), so the data
layer exposes one daily-feature schema that CovIdentify/Evidation drop into later.

**Repo conventions followed:** checkpoint keys `guarded_{component}_{user}_{timestamp}`,
warm-start chaining by `condition_key`, and the fixed frequency-tensor form
`torch.tensor([self.freq], dtype=torch.long)` — the scalar form in `beam_timesfm_v9.ipynb`
cell 27 is a known bug; do not copy it.

Run top to bottom. §1 setup · §2 data · §3 contamination harness · §4 detector ·
§5 LLM roles · §6 shield · §7 RL agent · §8 run loop · §9 frontier · §10 ablations · §11 checkpoints.


In [ ]:
import os, io, csv, zipfile, math, random, collections, statistics, json, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 7
rng = np.random.default_rng(SEED); random.seed(SEED); np.random.seed(SEED)

# ---- knobs (match the one-pager) ----
N_USERS     = 8
DAYS        = 140
LABEL_DELAY = 5      # days until a survey/diagnosis label is revealed
ASK_BUDGET  = 2      # LLM-written questions per episode (hard setting)
ALPHA       = 0.05   # shield significance: P(learn | illness) <= ALPHA
LAMBDA      = 3.0    # contamination penalty weight in the delayed reward
FLAG_Z      = 1.5    # detector flag threshold (robust z); shield band q sits just above it
WARMUP      = 7      # quiet days before any decision (detector warm-start)
GAMMA       = 0.95
USE_PBRS    = True   # LLM prior as potential-based reward shaping (one-pager 2c)

DATA_SOURCE  = os.environ.get('GP_DATA', 'simulator')  # simulator | stanford | covidentify | evidation
STANFORD_ZIP = os.environ.get('STANFORD_ZIP', 'COVID-19-Wearables.zip')

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
print(f'torch available: {HAS_TORCH}  |  data source: {DATA_SOURCE}')


## §2 Data layer — one daily-feature schema
`[user, day, rest_hr, hr_rest_std, steps, sleep_h, ill, benign, note]`.
`ill`: 1 = illness window, 0 = healthy. `benign`: 1 while a labelled healthy shift is active.
CovIdentify/Evidation loaders raise an informative error until credentialed — same schema when they land.


In [ ]:
COLS = ['user','day','rest_hr','hr_rest_std','steps','sleep_h','ill','benign','note']

def _synth_healthy(uid, days, seed):
    r = np.random.default_rng(seed)
    base = r.normal(62, 5)
    drift = np.linspace(0, r.normal(0, 2), days)
    weekly = 1.5*np.sin(2*np.pi*np.arange(days)/7)
    rest_hr = base + drift + weekly + r.normal(0, 1.2, days)
    return pd.DataFrame({
        'user': uid, 'day': np.arange(days),
        'rest_hr': np.round(rest_hr, 1),
        'hr_rest_std': np.round(np.abs(r.normal(3.2, 0.9, days)), 2),
        'steps': np.round(r.lognormal(8.6, 0.55, days)).astype(int),
        'sleep_h': np.round(r.normal(7.1, 1.0, days).clip(3.5, 10.5), 2),
        'ill': 0, 'benign': 0, 'note': ''})

def _load_stanford(zip_path, max_users=12):
    """Compact form of the verified prototype (2026-10-07): rest_hr = mean HR over
    minutes with <=20 steps/min; steps winsorised per user; sleep_h NaN when missing."""
    zf = zipfile.ZipFile(zip_path)
    uids = sorted({n.split('/')[-1].split('_')[0] for n in zf.namelist()
                   if n.endswith('.csv')})[:max_users]
    frames = []
    for uid in uids:
        def read(mod):
            try:
                with zf.open(f'COVID-19-Wearables/{uid}_{mod}.csv') as f:
                    return list(csv.DictReader(io.TextIOWrapper(f, encoding='utf-8', errors='replace')))
            except KeyError:
                return []
        hr = read('hr') or read('hr_longterm'); steps = read('steps'); sleep = read('sleep')
        if not hr or not steps: continue
        step_min = {row['datetime'][:16]: int(row['steps'] or 0) for row in steps}
        by_day = collections.defaultdict(lambda: {'hr_rest': [], 'steps': 0})
        for row in hr:
            d, key = row['datetime'][:10], row['datetime'][:16]
            s = step_min.get(key, 0); by_day[d]['steps'] += s
            if s <= 20:
                try: by_day[d]['hr_rest'].append(float(row['heartrate']))
                except (ValueError, TypeError): pass
        slp = collections.defaultdict(float)
        for row in sleep:
            if row['stage'] == 'wake': continue
            slp[row['datetime'][:10]] += float(row['stage_duration'])/3600.0
        rows = []
        for i, d in enumerate(sorted(by_day)):
            rest = by_day[d]['hr_rest']
            rows.append({'user': uid, 'day': i,
                'rest_hr': round(statistics.mean(rest), 1) if rest else np.nan,
                'hr_rest_std': round(statistics.pstdev(rest), 2) if len(rest) > 1 else np.nan,
                'steps': by_day[d]['steps'],
                'sleep_h': round(slp[d], 2) if d in slp else np.nan,
                'ill': 0, 'benign': 0, 'note': ''})
        f = pd.DataFrame(rows).dropna(subset=['rest_hr'])
        lo, hi = f['steps'].quantile(0.01), f['steps'].quantile(0.99)  # verified quirk: ~60k steps/day users
        f['steps'] = f['steps'].clip(lo, hi).astype(int)
        frames.append(f)
    zf.close()
    return pd.concat(frames, ignore_index=True) if frames else None

def load_daily_features(source='simulator', n_users=N_USERS, days=DAYS, seed=SEED, verbose=True):
    def _log(m):
        if verbose: print(m)
    if source == 'stanford' and os.path.exists(STANFORD_ZIP):
        df = _load_stanford(STANFORD_ZIP, max_users=n_users)
        if df is not None:
            _log(f'stanford: {df["user"].nunique()} users, {len(df)} rows'); return df[COLS]
        _log('stanford zip unreadable -> synthetic fallback')
    if source in ('covidentify', 'evidation'):
        raise RuntimeError(
            f"source='{source}' needs credentialed access (CovIdentify: PhysioNet credentialing + DUA; "
            "Evidation: Synapse request). Run with DATA_SOURCE=simulator until then.")
    if source not in ('simulator', 'stanford'):
        raise ValueError(f'unknown source {source}')
    df = pd.concat([_synth_healthy(f'u{i:02d}', days, seed+i) for i in range(n_users)],
                   ignore_index=True)
    _log(f'simulator: {n_users} users x {days} days (synthetic healthy traces)')
    return df[COLS]

base = load_daily_features(DATA_SOURCE)
base.head(3)


## §3 Contamination harness — labelled episodes with known ground truth
Each user gets one **benign shift** (device swap: +4 bpm offset, or routine change: 1.6× steps —
the model *should* learn these) and one **illness episode** (flu-like ramp: rest HR +0→15 bpm
over 3 days, fragmented sleep, collapsed steps — the model must *never* learn these).
A benign travel bump (one-pager Day-3 example) should be WAITed out.


In [ ]:
def inject_illness(df, uid, onset, dur=7, note='tired, sore throat', note_prob=0.6):
    """Flu-like episode with KNOWN onset (survey note present w.p. note_prob)."""
    m = (df['user'] == uid).to_numpy(); days = df.loc[m, 'day'].to_numpy()
    for i, d in zip(np.where(m)[0], days):
        if onset <= d < onset + dur:
            ramp = min(1.0, (d - onset + 1) / 3.0)
            df.at[i, 'rest_hr'] = round(df.at[i, 'rest_hr'] + 15*ramp, 1)
            df.at[i, 'sleep_h'] = round(df.at[i, 'sleep_h']*0.7, 2)
            df.at[i, 'steps']   = int(df.at[i, 'steps']*0.4)
            df.at[i, 'ill'] = 1
            df.at[i, 'note'] = note if random.random() < note_prob else ''
    return df

def inject_benign_shift(df, uid, at, kind='device', note=''):
    m = (df['user'] == uid).to_numpy(); days = df.loc[m, 'day'].to_numpy()
    for i, d in zip(np.where(m)[0], days):
        if d >= at:
            if kind == 'device': df.at[i, 'rest_hr'] = round(df.at[i, 'rest_hr'] + 4.0, 1)
            else:                df.at[i, 'steps']   = int(df.at[i, 'steps']*1.6)
            df.at[i, 'benign'] = 1
            if note: df.at[i, 'note'] = note
    return df

def build_cohort(df):
    df = df.copy()
    for j, uid in enumerate(sorted(df['user'].unique())):
        n = int((df['user'] == uid).sum())
        at_b, onset = min(40, n//3), min(70, n//2)
        df = inject_benign_shift(df, uid, at=at_b, kind='device' if j % 2 == 0 else 'routine')
        df = inject_illness(df, uid, onset=onset, dur=7)
        m = (df['user'] == uid).to_numpy(); days = df.loc[m, 'day'].to_numpy()
        for i, d in zip(np.where(m)[0], days):
            if onset - 50 <= d < onset - 47:   # benign travel bump -> WAIT it out
                df.at[i, 'rest_hr'] = round(df.at[i, 'rest_hr'] + 8, 1)
                df.at[i, 'note'] = 'flew to Europe, had wine'
    return df

data = build_cohort(base)
print('illness windows:', int(data['ill'].sum()),
      '| benign-shift days:', int(data['benign'].sum()))
data[data['ill'] == 1].head(3)


## §4 Detector — personal TimesFM + LoRA (interface) with a numpy fallback
The torch path mirrors repo conventions: warm-start chaining by `condition_key` and
checkpoint keys `guarded_detector_{user}_{condition}`. The numpy rolling-baseline fallback
runs anywhere (no GPU), so the mechanism tests execute end-to-end today.


In [ ]:
class PersonalDetector:
    def __init__(self, win=21):
        self.win = win; self.ckpt = {}
    def _key(self, user, condition):
        return f'guarded_detector_{user}_{condition}'
    def predict(self, user, history):
        h = np.asarray(history[-self.win:]); h = h[~np.isnan(h)]
        return float(np.median(h)) if len(h) else 65.0
    def spread(self, user, history):
        h = np.asarray(history[-self.win:]); h = h[~np.isnan(h)]
        return float(1.4826*np.median(np.abs(h - np.median(h)))) if len(h) > 3 else 3.0
    def anomaly_score(self, user, rest_hr, history):
        return (rest_hr - self.predict(user, history)) / max(self.spread(user, history), 1e-3)
    def adapt(self, user, window, condition_key='healthy'):
        # warm-start chaining: fit starts from the previous checkpoint for (user, condition)
        k = self._key(user, condition_key)
        self.ckpt[k] = {'n': self.ckpt.get(k, {'n': 0})['n'] + len(window)}
    # NOTE (repo bug, do not copy): beam_timesfm_v9.ipynb cell 27 builds
    #   freq = torch.tensor(self.freq, dtype=torch.long)      # scalar -> batching breaks
    # the working form is  torch.tensor([self.freq], dtype=torch.long)  # shape [1]

det = PersonalDetector()


## §5 LLM — three roles (one-pager §2)
**(a) Context reader** — parses survey text the sensors miss. **(b) Safety blocker** —
asymmetric: it may *close* the gate, never force it open. **(c) PBRS prior** — potential-based
reward shaping so RL learns from few illness examples (thesis Ch. 5). Trust in the LLM's
advice is learned from delayed outcomes (Beta-Bernoulli). Rule-based fallback keeps the
notebook runnable offline; set `GEMINI_API_KEY` to wire the real model.


In [ ]:
ILL_WORDS = ['flu','sick','fever','covid','sore throat','tired','ill','nausea']
OK_WORDS  = ['wine','travel','flew','exercise','workout','stress','alcohol','vacation','new watch']

class LLMAdvisor:
    def __init__(self, api_key=None):
        self.api_key = api_key or os.environ.get('GEMINI_API_KEY')
        self.a, self.b = 1.0, 1.0
    @property
    def trust(self): return self.a / (self.a + self.b)
    def _rule_read(self, note):
        n = (note or '').lower()
        si, so = sum(w in n for w in ILL_WORDS), sum(w in n for w in OK_WORDS)
        if si == 0 and so == 0: return 0.5, 'no context'
        return si/(si+so), f'keywords ill={si} ok={so}'
    def read_context(self, note, vitals=''):
        # TODO: Gemini call when api_key present; rule fallback offline
        p, why = self._rule_read(note)
        return {'p_ill': p, 'rationale': why, 'trust': self.trust}
    def safety_block(self, ctx):
        return ctx['p_ill'] > 0.6            # may veto ADAPT; can never approve it
    def pbrs_potential(self, p_bin):
        return p_bin/2 - 0.5                 # illness-like context -> negative potential
    def ask_question(self, rest_hr, ctx):
        q = f'Your resting heart rate has been elevated ({rest_hr:.0f} bpm). '
        q += ('Have you felt ill in the last few days?' if ctx['p_ill'] > 0.5
              else 'Have you started exercising, travelled, or changed device?')
        return q  # TODO: Gemini-written question when key present
    def update_trust(self, p_ill, true_ill):
        agree = (p_ill > 0.5) == bool(true_ill)
        self.a += float(agree); self.b += float(not agree)

llm = LLMAdvisor()
print('trust prior:', round(llm.trust, 3))
print(llm.read_context('tired, sore throat'), '| block =', llm.safety_block(llm.read_context('tired, sore throat')))
print(llm.read_context('flew to Europe, had wine'))


## §6 Safety shield — independent conformal gate
Split-conformal on known-healthy calibration windows: `ADAPT` is allowed only when
`|score|` sits inside the (1−α) band **and** the LLM has not blocked it.
Neither the RL policy nor the LLM can force the gate open.


In [ ]:
class ConformalShield:
    def calibrate(self, scores, alpha=ALPHA):
        s = np.abs(np.asarray(scores)); s = s[~np.isnan(s)]
        self.q = float(np.quantile(s, 1-alpha)) if len(s) else np.inf
    def allow_adapt(self, score, llm_block):
        # two-sided band: mild windows may pass, extreme ones never do;
        # the LLM block is asymmetric on top (may veto, never approve)
        return bool(abs(score) <= self.q and not llm_block)

def calibrate_shield(df, alpha=ALPHA):
    sh = ConformalShield(); d0 = PersonalDetector(); cal = []
    for uid, g in df.groupby('user'):
        g = g.sort_values('day'); hist = []
        for _, r in g.iloc[:15].iterrows():
            cal.append(d0.anomaly_score(uid, r['rest_hr'], hist)); hist.append(r['rest_hr'])
    sh.calibrate(cal, alpha); return sh

shield = calibrate_shield(data)
print(f'shield q(1-alpha={1-ALPHA}): {shield.q:.2f}')


## §7 RL agent — ALERT / ADAPT / WAIT / ASK
Tabular Q-learning over a discretized state (score × LLM advice × trust × budget).
Delayed reward (one-pager §3): benefit of healthy adaptation − λ·illness learned
− false alerts − questions − delay. Labels arrive `LABEL_DELAY` days late via a pending
buffer; ASK pays a question cost and reveals the label now (noisy user).


In [ ]:
ALERT, ADAPT, WAIT, ASK = 0, 1, 2, 3
ANAMES = ['ALERT', 'ADAPT', 'WAIT', 'ASK']

class RLAgent:
    def __init__(self, lr=0.2, eps=0.15):
        self.Q = collections.defaultdict(lambda: np.zeros(4))
        self.lr, self.eps = lr, eps
    @staticmethod
    def discretize(score, p_ill, trust, budget):
        return (min(2, int(max(0, score)//2)), min(2, int(p_ill*3)),
                int(trust > 0.5), int(budget > 0))
    def act(self, s, greedy=False):
        if not greedy and random.random() < self.eps: return random.randrange(4)
        return int(np.argmax(self.Q[s]))
    def update(self, s, a, r, s2):
        q = self.Q[s]; tgt = r + GAMMA*np.max(self.Q[s2])
        q[a] += self.lr*(tgt - q[a])

def rule_act(score, p_ill, shield_ok, budget):
    # strong shielded rule baseline for the October go/no-go
    if p_ill > 0.6: return ALERT
    if shield_ok and abs(score) <= 2.0: return ADAPT   # mild flagged windows only
    if budget > 0 and abs(score) > 3.0: return ASK
    return WAIT


## §8 Run loop — one episode over the cohort
For each flagged window the policy acts; `ADAPT` outcomes are settled when the label
is revealed `LABEL_DELAY` days later (pending buffer). LLM trust updates both on ASK
(immediate noisy answer) and on delayed reveals.


In [ ]:
def run_episode(agent, llm, det, shield, df, train=True, label_delay=LABEL_DELAY,
                ask_budget=ASK_BUDGET, lam=LAMBDA, use_shield=True, use_llm=True, rule=False,
                learn_trust=True):
    stats = collections.Counter(); first_alert = {}
    for uid, g in df.groupby('user'):
        pending = collections.deque()   # per-user: a stale head must not block later users
        g = g.sort_values('day').reset_index(drop=True)
        onset = int(g.loc[g['ill'] == 1, 'day'].min()) if (g['ill'] == 1).any() else None
        hist, budget = [], ask_budget
        for t, r in g.iterrows():
            score = det.anomaly_score(uid, r['rest_hr'], hist)
            while pending and pending[0][3] <= t:   # delayed labels revealed (every day)
                ps, pa, t0, _, pp = pending.popleft()
                row = g.iloc[t0]
                realised = (1.0 if (row['benign'] == 1 and row['ill'] == 0)
                            else (-lam if row['ill'] == 1 else -0.1))
                if row['ill'] == 1: stats['contaminated'] += 1      # learned illness: the failure
                elif row['benign'] == 1: stats['useful_adapt'] += 1  # learned healthy shift: the goal
                else: stats['neutral_adapt'] += 1                    # harmless churn
                if use_llm and learn_trust: llm.update_trust(pp, row['ill'])
                if train and agent is not None and not rule:
                    ps2 = agent.discretize(score, pp, llm.trust, budget)
                    F = (GAMMA*llm.pbrs_potential(ps2[1]) - llm.pbrs_potential(ps[1])) if (USE_PBRS and use_llm and ps is not None) else 0.0
                    agent.update(ps, pa, realised + F, ps2)
            if t < WARMUP or abs(score) < FLAG_Z:
                hist.append(r['rest_hr']); continue
            ctx = (llm.read_context(r['note']) if use_llm
                   else {'p_ill': 0.5, 'rationale': 'llm off', 'trust': llm.trust})
            blocked = llm.safety_block(ctx) if use_llm else False
            s = agent.discretize(score, ctx['p_ill'], ctx['trust'], budget) if agent else None
            if rule:
                ok = shield.allow_adapt(score, blocked) if use_shield else True
                a = rule_act(score, ctx['p_ill'], ok, budget)
            else:
                a = agent.act(s, greedy=not train)
            if a == ADAPT and use_shield and not shield.allow_adapt(score, blocked):
                a = WAIT; stats['shield_vetoes'] += 1
            if a == ASK and (budget <= 0 or not use_llm):
                a = WAIT
            imm = 0.0
            if a == ALERT:
                imm = 1.0 if r['ill'] == 1 else -0.5
                stats['alerts'] += 1
                if r['ill'] == 1:
                    stats['true_alerts'] += 1
                    if uid not in first_alert: first_alert[uid] = (t, onset)
                else: stats['false_alerts'] += 1
            elif a == ADAPT:
                det.adapt(uid, [r['rest_hr']])
                pending.append((s, a, t, t + label_delay, ctx['p_ill']))
                stats['adapts'] += 1
            elif a == ASK:
                budget -= 1; imm = -0.2; stats['asks'] += 1
                revealed = r['ill'] if random.random() < 0.85 else 1 - r['ill']
                if use_llm and learn_trust: llm.update_trust(ctx['p_ill'], revealed)
            else:
                imm = -0.01; stats['waits'] += 1
            if train and agent is not None and not rule and a in (ALERT, ASK, WAIT):
                s2 = agent.discretize(score, ctx['p_ill'], ctx['trust'], budget)
                F = (GAMMA*llm.pbrs_potential(s2[1]) - llm.pbrs_potential(s[1])) if (USE_PBRS and use_llm) else 0.0
                agent.update(s, a, imm + F, s2)
            hist.append(r['rest_hr'])
    return stats, first_alert


## §9 Train, then the contamination–adaptation frontier
Train the RL policy for 30 episodes, then sweep the shield strictness α at eval time:
tighter α → fewer adaptations, less contamination. That sweep *is* the trade-off curve
from the one-pager.


In [ ]:
def evaluate(agent, llm, n_ep=4, alpha=ALPHA, seed0=1000, **kw):
    tot = collections.Counter(); delays = []
    for e in range(n_ep):
        df = build_cohort(load_daily_features('simulator', N_USERS, DAYS,
                                              seed=seed0+e, verbose=False))
        sh = calibrate_shield(df, alpha)
        st, fa = run_episode(agent, llm, PersonalDetector(), sh, df, train=False,
                          learn_trust=False, **kw)   # frozen trust at eval: clean ablations
        for k, v in st.items(): tot[k] += v
        for uid, (t, onset) in fa.items():
            if onset is not None: delays.append(t - onset)
    tot['mean_delay'] = float(np.mean(delays)) if delays else float('nan')
    tot['n_detected'] = len(delays)
    return tot

agent = RLAgent(); llm = LLMAdvisor()
for ep in range(30):
    df = build_cohort(load_daily_features('simulator', N_USERS, DAYS, seed=SEED+ep, verbose=False))
    run_episode(agent, llm, PersonalDetector(), calibrate_shield(df), df, train=True)
print('trained. LLM trust:', round(llm.trust, 3))

res = evaluate(agent, llm, n_ep=6)
for k in ['adapts','useful_adapt','contaminated','neutral_adapt','alerts','true_alerts','false_alerts','asks','waits','shield_vetoes']:
    print(f'{k:15s} {int(res.get(k, 0))}')
print(f'mean detection delay: {res["mean_delay"]:.1f} days | users detected: {res["n_detected"]}/{N_USERS*6}')


In [ ]:
pts = []
for a in [0.01, 0.05, 0.2, 0.5]:
    r = evaluate(agent, llm, n_ep=4, alpha=a, seed0=2000)
    pts.append((a, r['useful_adapt'], r['contaminated']))
    print(f'alpha={a:<4} useful_adapt={int(r["useful_adapt"]):3d}  contaminated={int(r["contaminated"]):3d}')
plt.figure(figsize=(6, 4))
plt.scatter([p[1] for p in pts], [p[2] for p in pts], s=80)
for a, x, y in pts: plt.annotate(f'alpha={a}', (x, y), textcoords='offset points', xytext=(6, 4))
plt.xlabel('useful adaptations (benign shifts learned)'); plt.ylabel('contaminated windows (illness learned)')
plt.title('Contamination–adaptation frontier (shield strictness sweep)'); plt.grid(alpha=0.3); plt.show()


## §10 Ablations — does each part earn its place?
Lesion study on the trained policy: no shield, no LLM, and the strong shielded rule
baseline from the October go/no-go. (October extensions: shuffled-LLM advice,
frozen-vs-learned trust, BERT-vs-LLM — hooks are the `use_llm`/trust flags above.)


In [ ]:
configs = {
    'full (rl+shield+llm)': dict(agent=agent, use_shield=True,  use_llm=True,  rule=False),
    'no shield':            dict(agent=agent, use_shield=False, use_llm=True,  rule=False),
    'no llm':               dict(agent=agent, use_shield=True,  use_llm=False, rule=False),
    'rule baseline':         dict(agent=None,  use_shield=True,  use_llm=True,  rule=True),
}
rows = []
for name, kw in configs.items():
    L = LLMAdvisor()
    r = evaluate(kw.pop('agent'), L, n_ep=4, seed0=3000, **kw)
    rows.append((name, int(r['useful_adapt']), int(r['contaminated']),
                 int(r['true_alerts']), int(r['false_alerts']), int(r['asks'])))
    print(f'{name:22s} useful={rows[-1][1]:3d} contam={rows[-1][2]:3d} '
          f'true_alert={rows[-1][3]:3d} false_alert={rows[-1][4]:3d} asks={rows[-1][5]:3d}')
lab = [x[0] for x in rows]; x = np.arange(len(lab)); w = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x-w/2, [v[1] for v in rows], w, label='useful adaptations')
plt.bar(x+w/2, [v[2] for v in rows], w, label='contaminated')
plt.xticks(x, lab, rotation=12, ha='right'); plt.legend(); plt.grid(alpha=0.3, axis='y')
plt.title('Ablations: contamination vs useful adaptation'); plt.tight_layout(); plt.show()


## §11 Checkpoints — repo key conventions
`guarded_{component}_{user}_{timestamp}` (mirrors `timesfm_lora_{method}_{dataset}_b{NN}_{ts}.pth`).
Per-user detector keys follow `guarded_detector_{user}_{condition}` (see §4).


In [ ]:
ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
os.makedirs('checkpoints', exist_ok=True)
ckpt = {'Q': {','.join(map(str, k)): v.tolist() for k, v in agent.Q.items()},
        'trust_a': llm.a, 'trust_b': llm.b,
        'alpha': ALPHA, 'lambda': LAMBDA, 'label_delay': LABEL_DELAY,
        'ask_budget': ASK_BUDGET, 'seed': SEED, 'has_torch': HAS_TORCH}
path = f'checkpoints/guarded_rl_policy_all_{ts}.json'
json.dump(ckpt, open(path, 'w'))
print('saved', path, f'({len(ckpt["Q"])} states)')


## Next steps (her TODOs, not the notebook's)
1. **TimesFM path** — with torch + GPU, swap the numpy fallback in §4 for the real
   TimesFM+LoRA (warm-start by `condition_key`; per-user keys as in §11).
2. **Gemini** — set `GEMINI_API_KEY`; wire `read_context` / `ask_question` in §5
   (the repo's `gemini_judge_cache.json` pattern).
3. **Real labels** — `load_daily_features('covidentify' | 'evidation')` once credentialed;
   the schema is identical, so §§3–10 run unchanged.
4. **October go/no-go** — hard settings (label delay sweep, ASK budget 2, shuffled LLM
   advice, LLM reliability drift) using the ablation hooks in §10.
